# 12.4 Generating the Ground Truth

The enriched scenarios are inputs; scoring them needs an answer key whose own correctness is not in question. That key is *generated*, not annotated -- read from the same geometric memory substrate (the GEODE store) that the agent runs on. The same store is both what the agent consults and what the test scores it against, which is how the ground truth stays provably correct rather than a second opinion.

Before any score built on this key is trusted, there is a **precondition**: the graph must answer its own generated questions perfectly -- a baseline of `1.0`. A perfect baseline confirms the ground truth is sound; only then is a failure on the real run chargeable to the agent rather than to a noisy key.

**Reference (run separately):** the GEODE self-correction loop ingests the policy, proposes facts, diagnoses the ones the geometry finds implausible or contradictory, repairs them, then trains and calibrates the result. The build below produces the store the answer key is read from; the precondition is that the graph then answers its own generated questions at a baseline of `1.0`.

```python
# scripts/build_geode_rag_store.py -> data/gms_policy_store_geode
from knowlytix.geode import GeodeBuilder

builder = GeodeBuilder.from_policy('data/banking_policy_full.md')
store = builder.ingest().diagnose().repair().train().calibrate()
store.save('data/gms_policy_store_geode')   # the GEODE oracle

# The precondition: the graph answers its own generated questions perfectly.
from knowlytix.harness.testing import CapstoneTestHarness
h = CapstoneTestHarness(store='data/gms_policy_store_geode')
baseline = h.substrate_test().baseline      # must be 1.0 before any score is trusted
assert baseline == 1.0, 'ground truth is not sound; do not score the agent'
```

**What the next cell does** — finds the project root and loads the pinned companion artifact:

1. **Walk up to the root.** Starts at `Path.cwd()` and climbs parent directories until `data/capstone_companions.json` exists, so the cell runs from anywhere in the tree.
2. **Load the companions.** Reads `capstone_companions.json` into `C` and pulls `C['substrate']` into `substrate`, the block holding the ground-truth precondition numbers.

In [ ]:
import json
from pathlib import Path

root = Path.cwd().parent / 'code'
while not (root / 'data' / 'capstone_companions.json').exists() and root != root.parent:
    root = root.parent

C = json.loads((root / 'data' / 'capstone_companions.json').read_text())
substrate = C['substrate']
print('loaded capstone_companions.json from', root / 'data')

**What the next cell does** — prints the ground-truth precondition and renders a verdict:

1. **Read the numbers.** Pulls `substrate['baseline']` and `substrate['n_questions']` into `baseline` and `n_questions`.
2. **Print the baseline table.** Reports how many questions were reversed from the policy graph and the precondition baseline, formatted to two decimals.
3. **Render the verdict.** Sets `verdict` to `SOUND` when `baseline == 1.0` and `NOISY KEY` otherwise, then prints it.

In [ ]:
baseline = substrate['baseline']
n_questions = substrate['n_questions']

print('Ground-truth precondition (graph answers its own questions)')
print('-' * 56)
print(f'{"questions reversed from the policy graph":42s} {n_questions}')
print(f'{"precondition baseline (must equal 1.0)":42s} {baseline:.2f}')
print()
verdict = 'SOUND -> agent failures are chargeable' if baseline == 1.0 else 'NOISY KEY -> do not score'
print(f'ground truth: {verdict}')

A perfect baseline of `1.0` confirms the answer key is sound: the governing policy and its accept-set are graph-connected, the regulation's severity is a graph traversal, the authoritative fee is the exact-memory value, and the substrate questions are reversed wholesale out of the policy graph with provable answers. The seed cases' classification labels are the one authored input; everything fact-grounded is read from the graph.

This is the condition that makes the capstone a *designed experiment* rather than a benchmark: only once the key answers its own questions perfectly is a failure on the real run attributable to the agent rather than to a noisy ground truth.

**What the next cell does** — asserts the answer key is sound before any scoring is trusted:

1. **Check the block exists.** Asserts `'substrate'` is present in `C`.
2. **Check the baseline.** Asserts `substrate['baseline']` is exactly `1.0`.
3. **Check the question count.** Asserts `n_questions > 0`, then prints `self-check passed`.

In [ ]:
assert 'substrate' in C, 'companions artifact must carry the substrate block'
assert substrate['baseline'] == 1.0, 'precondition baseline must be exactly 1.0'
assert n_questions > 0, 'questions must be generated from the graph'
print('self-check passed')